## Code to convert the pickle files to csv 

Pickle files are non-trivial to open on versions of pandas that it was not saved in. 
Therefore with this code I convert the pickle files with the stats of bursts to csv files and at the end merge the in a final stats.csv file. <br>

There are three intermediate steps: <br>
    - Pband-bursts - completed: 18-Jul-23 <br>
    - Stockert-bursts - completed: 05-Jul-23 <br>
    - EVN L-band bursts - completed: 01-Aug-23 <br>
    - Recalculated the arrival times for 2 stk bursts - completed: 19-Dec-23 <br>
Good version 1: 01-Aug-23

In [1]:
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen

In [2]:
## Function ran on 18-Jul
def pband_pickle(file):
    
    df = pd.read_pickle(file)
    
    #Filter on Westerbork Only, since there might be a stockert burst in there
    df_wb = df[df['dish'] == 'wb']
    df_wb = df_wb.sort_values(by="id",key=natsort_keygen())
    
    df_wb = df_wb.reset_index()
    df_wb = df_wb.drop('index', axis=1)
    
    #Assign values to the empty columns
    df_wb = df_wb.assign(dm=219.7)
    df_wb = df_wb.assign(distance_Mpc=362.4)
    df_wb = df_wb.assign(sefd_jy=2100)
    
    bursts = df_wb['id'].unique()
    print(len(bursts), bursts)
    
    mjds = df_wb["toa_bary_tdb_inf_freq"].values
    mjds_sorted = mjds[mjds!=0]
    mjds_sorted.sort()
    
    print(mjds_sorted)
    
    
    display(df_wb)
    
    #save the csv information in the git
    name_csv = 'burst_info_wb_pband_16.csv'
    df_wb.to_csv(name_csv, header=True, index=True, na_rep='NA')
        
    return name_csv
pname = pband_pickle(file="burst_info_P.pickle")

16 ['B100-wb' 'B109-wb' 'B110-wb' 'B115-wb' 'B117-wb' 'B118-wb' 'B119-wb'
 'B120-wb' 'B121-wb' 'B122-wb' 'B123-wb' 'B124-wb' 'B125-wb' 'B126-wb'
 'B127-wb' 'B128-wb']
[59905.887227454914 59911.88364480753 59912.980998424304
 59922.820334706186 59931.627348594615 59935.88899182841 59936.69746185126
 59939.628285379295 59939.86096635976 59939.86096639343 59939.86096646742
 59939.873744337034 59940.62894168331 59940.66155953371 59940.85143235003
 59941.650576342705 59941.68738958076 59941.730175512756 59941.73017559139
 59941.730175670746 59941.73017574266]


,experiment,dish,scan,component,src,t0_ms,f0_MHz,toa_loc,terr_ms,ferr_MHz,...,scint_bw_err_MHz,energy_iso_erg_per_hz,spectral_lum_erg_per_s_per_hz,id,sefd_jy,distance_Mpc,dm,toa_mjd_utc,ref_freq_MHz,toa_bary_tdb_inf_freq
0,p11714,wb,no0035,0,spc,1935.15773,328.0,0,0,0,...,1.875832,71289870960556706039858930384896.0,1450396137706638669301746078056448.0,B100-wb,2100,362.4,219.7,0,0,0
1,p11714,wb,no0035,0,scale,1935.15773,328.0,0,0,0,...,1.363507,72325825748228021788850371166208.0,1471472691817790172810667232329728.0,B100-wb,2100,362.4,219.7,0,0,0
2,p11714,wb,no0035.ds.pazi,0,sfxc,1016.642768,328.0,coe,0,0,...,1.636118,75344635778153154349869790396416.0,1313906176376833849858095312797696.0,B100-wb,2100,362.4,219.7,59905.883816,352.0,59905.887227
3,p11720,wb,no0036,0,spc,1965.115174,328.0,0,0,0,...,19.675118,44316815518976251142459189362688.0,1081953503881256285867088522772480.0,B109-wb,2100,362.4,219.7,0,0,0
4,p11720,wb,no0036,0,scale,1965.115174,328.0,0,0,0,...,71833.166692,44846511510440203241600785055744.0,1094885534922856677153319031930880.0,B109-wb,2100,362.4,219.7,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,p11749,wb,no0030,2,scale,2029.560904,328.0,0,0,0,...,0.097566,102109927767134462538574857240576.0,16619454389182040121634709659713536.0,B128-wb,2100,362.4,219.7,0,0,0
59,p11749,wb,no0030,3,scale,2036.511786,328.0,0,0,0,...,0.240733,144855827178801177717871917137920.0,11788397394108168043144800452476928.0,B128-wb,2100,362.4,219.7,0,0,0
60,p11749,wb,no0030,1,spc,2022.488077,328.0,0,0,0,...,0.106317,124659184371447529596822889693184.0,17391069248248818549539430071271424.0,B128-wb,2100,362.4,219.7,0,0,0
61,p11749,wb,no0030,3,spc,2036.511786,328.0,0,0,0,...,0.241404,143195131212454306987704903008256.0,11653249610388534753424884824539136.0,B128-wb,2100,362.4,219.7,0,0,0


In [3]:
## Function ran on 05-Jul
##Function to convert the Stockert pd dataframe to a csv-file
def stockert_pickle(file):
    
    df = pd.read_pickle(file)
    
    #Filter only on Stockert
    df_stk = df[df['dish'] == 'st']
    
    df_stk = df_stk.sort_values(by="id",key=natsort_keygen())
    
    #Reset the index
    df_stk = df_stk.reset_index()
    
    #Drop the index column
    df_stk = df_stk.drop('index', axis=1)
    
    #Assign values to the empty columns
    df_stk = df_stk.assign(dm=219.37)
    df_stk = df_stk.assign(distance_Mpc=362.4)
    df_stk = df_stk.assign(sefd_jy=385)
    
    bursts = df_stk['id'].unique()
    print(len(bursts), bursts)
    
    
    display(df_stk)
    
    name_csv = 'burst_info_stockert_85.csv'
    df_stk.to_csv(name_csv, header=True, index=True, na_rep='NA')
    
    return name_csv
stk_name = stockert_pickle(file="burst_info_V1.pickle")

85 ['B01-st' 'B02-st' 'B03-st' 'B04-st' 'B07-st' 'B08-st' 'B09-st' 'B10-st'
 'B11-st' 'B12-st' 'B13-st' 'B17-st' 'B18-st' 'B19-st' 'B20-st' 'B25-st'
 'B26-st' 'B27-st' 'B28-st' 'B29-st' 'B30-st' 'B31-st' 'B32-st' 'B34-st'
 'B35-st' 'B36-st' 'B38-st' 'B39-st' 'B40-st' 'B41-st' 'B42-st' 'B43-st'
 'B47-st' 'B48-st' 'B49-st' 'B50-st' 'B53-st' 'B54-st' 'B56-st' 'B57-st'
 'B58-st' 'B59-st' 'B60-st' 'B61-st' 'B62-st' 'B63-st' 'B64-st' 'B65-st'
 'B67-st' 'B68-st' 'B69-st' 'B73-st' 'B74-st' 'B75-st' 'B76-st' 'B77-st'
 'B78-st' 'B79-st' 'B80-st' 'B82-st' 'B85-st' 'B86-st' 'B87-st' 'B88-st'
 'B89-st' 'B90-st' 'B91-st' 'B92-st' 'B93-st' 'B94-st' 'B95-st' 'B96-st'
 'B98-st' 'B99-st' 'B102-st' 'B104-st' 'B106-st' 'B107-st' 'B108-st'
 'B112-st' 'B113-st' 'B114-st' 'B116-st' 'B129-st' 'B131-st']


,experiment,dish,scan,component,src,t0_ms,f0_MHz,toa_loc,terr_ms,ferr_MHz,...,scint_bw_err_MHz,energy_iso_erg_per_hz,spectral_lum_erg_per_s_per_hz,id,sefd_jy,distance_Mpc,dm,toa_mjd_utc,ref_freq_MHz,toa_bary_tdb_inf_freq
0,stocke,st,5457019.ds.pazi,0,sfxc,215.237374,1381.464844,st,0,0,...,0.519943,5521452561158713250350335262720.0,1053133499366514871458652680617984.0,B01-st,385,362.4,219.37,59867.545702,1430.390625,59867.550262
1,stocke,st,5457019.ds.pazi,1,sfxc,220.515865,1381.464844,st,0,0,...,0.83416,1105063789741589023520125353984.0,459870961862011628071817893117952.0,B01-st,385,362.4,219.37,59867.545702,1430.390625,59867.550262
2,stocke,st,7161709.ds.pazi,0,sfxc,221.039969,1381.464844,st,0,0,...,897019.490481,3813137866377431047851460263936.0,484865553077371027631976017297408.0,B02-st,385,362.4,219.37,59868.716171,1430.390625,59868.720719
3,stocke,st,8024511.ds.pazi,0,sfxc,210.813678,1381.464844,st,0,0,...,0.348843,4673317429088140284082054496256.0,345044346157551917481591371202560.0,B03-st,385,362.4,219.37,59868.802451,1430.390625,59868.806998
4,stocke,st,8913731.ds.pazi,0,sfxc,218.868976,1381.464844,st,0,0,...,0.332759,1854127646414176947895743283200.0,404167752165478678007882480353280.0,B04-st,385,362.4,219.37,59868.947519,1430.390625,59868.952065
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141,stocke,st,2650104.ds.pazi,0,sfxc,225.598572,1381.464844,st,0,0,...,0.366147,4524171944773484263215707717632.0,1218236212725511612010787980181504.0,B113-st,385,362.4,219.37,59917.26501,1430.390625,59917.267816
142,stocke,st,5089328.ds.pazi,0,sfxc,216.428713,1381.464844,st,0,0,...,1.659692,22775501545267558104206147584000.0,5212898608078206618596177994579968.0,B114-st,385,362.4,219.37,59917.508933,1430.390625,59917.511725
143,stocke,st,0072834.ds.pazi,0,sfxc,216.249564,1381.464844,st,0,0,...,2.268304,2635846533103484845651518291968.0,354880819850717574741472242040832.0,B116-st,385,362.4,219.37,59925.007283,1430.390625,59925.009633
144,stocke,st,8819706.ds.pazi,0,sfxc,222.476264,1381.464844,st,0,0,...,5.693818,7253632282586246911335187087360.0,572491266915123731168990190370816.0,B129-st,385,362.4,219.37,59949.881971,1430.390625,59949.882698


In [4]:
## Function ran on 24-Jul
##Function to convert the european dishes pd dataframe to a csv-file
def euro_pickle(file):
    
    df = pd.read_pickle(file)
    
    #Filter on Eur dishes and remove the p-band entries that ended up in the pickle file
    df_eur = df[(df['dish'] != 'st') & (df['f0_MHz'] != 328.0)]
    
    #Reset the index
    df_eur = df_eur.reset_index()

    #We drop all rows from B130-tr, since the values can't be trusted. 
    df_eur.drop(df_eur.loc[df_eur['id']=="B130-tr"].index, inplace=True)
    
    #Add a manual entry here for the burst B130-tr
    df_130 = pd.DataFrame([[0, 'f22l24','tr','no0011.ds.pazi','sfxc','B130-tr']], \
                          columns=['index','experiment','dish','scan','src','id'])
    
    df_eur = pd.concat([df_eur, df_130], ignore_index=True)
    df_eur = df_eur.fillna(0)
    
    #Drop the index column
    df_eur = df_eur.drop('index', axis=1)
    
    #Assign values to the empty columns; valid on all entries
    df_eur = df_eur.assign(dm=219.37)
    df_eur = df_eur.assign(distance_Mpc=362.4)
    
    #If wb is in the line, then assign 420 jyms (wb) otherwise assign 250 (tr). 
    #If wb is true, then 420 will be assigned, otherwise it will be 250.
    df_eur["sefd_jy"] = np.where(df_eur["dish"].isin(["wb"]), 420, 250)
    jy = df_eur["sefd_jy"].unique()
    print(f"sefd values: {jy}")
    
    #Sort the pandas array by id, use natsort to do this
    #link: https://stackoverflow.com/questions/29580978/naturally-sorting-pandas-dataframe 
    df_eur = df_eur.sort_values(by="id",key=natsort_keygen())
    
    ##Double check if all SFXC-components in the db have a TOA
#     df_eur_empty = df_eur[(df_eur['ref_freq_MHz'] == 0) & (df_eur['src'] == 'sfxc')]
#     #SHould retunr empty
#     empty = df_eur_empty['id'].unique()
#     print(empty)
#     display(df_eur_empty)    
        
    #Display all burst that are in the array. 
    bursts = df_eur['id'].unique()
    print(len(bursts), bursts)
    print(df_eur['f0_MHz'].unique())
    
    display(df_eur)
    
    name_csv = 'burst_info_euro_46.csv'
    df_eur.to_csv(name_csv, header=True, index=True, na_rep='NA')
    
    return name_csv

euro_name = euro_pickle(file="burst_info_V1.pickle")

sefd values: [250 420]
46 ['B05-tr' 'B06-tr' 'B07-tr' 'B13-tr' 'B14-wb' 'B15-tr' 'B15-wb' 'B16-tr'
 'B16-wb' 'B21-tr' 'B22-wb' 'B23-wb' 'B24-wb' 'B30-wb' 'B33-wb' 'B37-tr'
 'B37-wb' 'B44-wb' 'B45-wb' 'B46-wb' 'B51-wb' 'B52-wb' 'B53-wb' 'B55-wb'
 'B63-wb' 'B64-wb' 'B66-wb' 'B68-wb' 'B69-wb' 'B70-wb' 'B71-wb' 'B72-wb'
 'B81-wb' 'B82-wb' 'B83-wb' 'B85-wb' 'B97-tr' 'B101-wb' 'B103-wb'
 'B104-wb' 'B105-wb' 'B106-wb' 'B107-wb' 'B111-tr' 'B130-tr' 'B132-tr']
[1414. 1271.    0.]


,experiment,dish,scan,component,src,t0_ms,f0_MHz,toa_loc,terr_ms,ferr_MHz,...,scint_bw_err_MHz,energy_iso_erg_per_hz,spectral_lum_erg_per_s_per_hz,id,sefd_jy,distance_Mpc,dm,toa_mjd_utc,ref_freq_MHz,toa_bary_tdb_inf_freq
0,r67l62,tr,no0011.ds.pazi,0,sfxc,249.570521,1414.0,coe,0,0,...,0.048579,5.264690e+30,1.371013e+33,B05-tr,250,362.4,219.37,59868.947519,1470.0,59868.952070
1,r67l62,tr,no0011,0,scale,998.069482,1414.0,0,0,0,...,0.096978,6.697433e+30,1.687861e+33,B05-tr,250,362.4,219.37,0.000000,0.0,0.000000
2,r67l62,tr,no0011,0,spc,998.069482,1414.0,0,0,0,...,0.092705,6.404666e+30,1.614079e+33,B05-tr,250,362.4,219.37,0.000000,0.0,0.000000
3,r67l62,tr,no0015.ds.pazi,0,sfxc,253.264006,1414.0,coe,0,0,...,0.082156,2.296931e+30,4.849939e+32,B06-tr,250,362.4,219.37,59868.980665,1470.0,59868.985215
4,r67l62,tr,no0015,0,scale,1005.104219,1414.0,0,0,0,...,0.629538,3.464727e+30,5.639205e+32,B06-tr,250,362.4,219.37,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,f22l16,tr,no0011,1,spc,995.355704,1414.0,0,0,0,...,0.441273,1.198975e+31,1.419242e+33,B111-tr,250,362.4,219.37,0.000000,0.0,0.000000
231,f22l24,tr,no0011.ds.pazi,0,sfxc,0.000000,0.0,0,0,0,...,0.000000,0.000000e+00,0.000000e+00,B130-tr,250,362.4,219.37,0.000000,0.0,0.000000
221,f22l46,tr,no0019.ds.pazi,0,sfxc,262.187996,1414.0,coe,0,0,...,0.110960,5.981899e+30,1.168340e+33,B132-tr,250,362.4,219.37,59982.720527,1470.0,59982.719181
222,f22l46,tr,no0019,0,scale,904.515625,1414.0,0,0,0,...,0.183644,8.564453e+30,1.239070e+33,B132-tr,250,362.4,219.37,0.000000,0.0,0.000000


In [5]:
print(pname, stk_name, euro_name)

burst_info_wb_pband_16.csv burst_info_stockert_85.csv burst_info_euro_46.csv


In [6]:
def merger(pband_csv, stk_csv, eu_csv):
    "Function the merge the .csv files from each telescopes into 1 big csv file"

    df_pband = pd.read_csv(pband_csv)
    df_stk = pd.read_csv(stk_csv)
    df_eu = pd.read_csv(eu_csv)
    
    df_all = pd.concat([df_pband, df_stk, df_eu])
    
    #Remove the columns that is made
    df_all = df_all.drop('Unnamed: 0', axis=1)
    
    df_all = df_all.sort_values(by="id",key=natsort_keygen())
    df_all = df_all.reset_index()
    df_all = df_all.drop('index', axis=1)
    
    display(df_all)
    display(df_all.columns)
    
    name_csv = 'burst_stats_r117.csv'
    df_all.to_csv(name_csv, header=True, index=True, na_rep='NA')
    
    return

merger(pband_csv=pname, stk_csv=stk_name, eu_csv=euro_name)

,experiment,dish,scan,component,src,t0_ms,f0_MHz,toa_loc,terr_ms,ferr_MHz,...,scint_bw_err_MHz,energy_iso_erg_per_hz,spectral_lum_erg_per_s_per_hz,id,sefd_jy,distance_Mpc,dm,toa_mjd_utc,ref_freq_MHz,toa_bary_tdb_inf_freq
0,stocke,st,5457019.ds.pazi,0,sfxc,215.237374,1381.464844,st,0,0,...,0.519943,5.521453e+30,1.053133e+33,B01-st,385,362.4,219.37,59867.545702,1430.390625,59867.550262
1,stocke,st,5457019.ds.pazi,1,sfxc,220.515865,1381.464844,st,0,0,...,0.834160,1.105064e+30,4.598710e+32,B01-st,385,362.4,219.37,59867.545702,1430.390625,59867.550262
2,stocke,st,7161709.ds.pazi,0,sfxc,221.039969,1381.464844,st,0,0,...,897019.490481,3.813138e+30,4.848656e+32,B02-st,385,362.4,219.37,59868.716171,1430.390625,59868.720719
3,stocke,st,8024511.ds.pazi,0,sfxc,210.813678,1381.464844,st,0,0,...,0.348843,4.673317e+30,3.450443e+32,B03-st,385,362.4,219.37,59868.802451,1430.390625,59868.806998
4,stocke,st,8913731.ds.pazi,0,sfxc,218.868976,1381.464844,st,0,0,...,0.332759,1.854128e+30,4.041678e+32,B04-st,385,362.4,219.37,59868.947519,1430.390625,59868.952065
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436,f22l24,tr,no0011.ds.pazi,0,sfxc,0.000000,0.000000,0,0,0,...,0.000000,0.000000e+00,0.000000e+00,B130-tr,250,362.4,219.37,0.000000,0.000000,0.000000
437,stocke,st,5334682.ds.pazi,0,sfxc,215.019836,1381.464844,st,0,0,...,0.354273,5.895145e+31,4.423907e+33,B131-st,385,362.4,219.37,59978.533468,1430.390625,59978.532358
438,f22l46,tr,no0019.ds.pazi,0,sfxc,262.187996,1414.000000,coe,0,0,...,0.110960,5.981899e+30,1.168340e+33,B132-tr,250,362.4,219.37,59982.720527,1470.000000,59982.719181
439,f22l46,tr,no0019,0,scale,904.515625,1414.000000,0,0,0,...,0.183644,8.564453e+30,1.239070e+33,B132-tr,250,362.4,219.37,0.000000,0.000000,0.000000


Index(['experiment', 'dish', 'scan', 'component', 'src', 't0_ms', 'f0_MHz',
       'toa_loc', 'terr_ms', 'ferr_MHz', 'gauss2d_twidth_ms',
       'gauss2d_fwidth_MHz', 'begin_acf_range', 'end_acf_range',
       'twidth_sigma_ms', 'fwidth_sigma_MHz', 'twidtherr_ms', 'fwidtherr_MHz',
       'off_range_t0', 'off_range_t1', 'fluence_jyms', 'peak_snr',
       'peak_flux_jy', 'fluence_tot_jyms', 'scint_bw_MHz', 'scint_bw_err_MHz',
       'energy_iso_erg_per_hz', 'spectral_lum_erg_per_s_per_hz', 'id',
       'sefd_jy', 'distance_Mpc', 'dm', 'toa_mjd_utc', 'ref_freq_MHz',
       'toa_bary_tdb_inf_freq'],
      dtype='object')